The objective here will be to use a bitmap font atlas to match characters in the text field, and from that get an accurate OCR.

1. Load Atlas
2. Load Images
3. For each image:
   1. Prep a strip of the image for the date

In [1]:
import numpy as np
import dask.array as da
import dask.dataframe as dd
import pandas as pd
import dask
from dask_image import ndfilters, ndmeasure, ndmorph

from PIL import Image
from dask.array.image import imread
filename_pattern = r'/home/rainybyte/AllSkyImages/2010-08/*.JPG'
images = imread(filename_pattern)
images

dask.array<imread, shape=(8595, 480, 640, 3), dtype=uint8, chunksize=(1, 480, 640, 3), chunktype=numpy.ndarray>

In [2]:
red = images.transpose(0, 3, 1, 2)[0][0]
red

dask.array<getitem, shape=(480, 640), dtype=uint8, chunksize=(480, 640), chunktype=numpy.ndarray>

In [3]:
# These are specific to my atlas
atlas_chars = "0123456789s"
atlas_symbs = ":/."
atlas_image = imread('/home/rainybyte/AllSkyImages/font_atlas.bmp') # 66 x 16 pixels
char_width = 6
char_height = 8
symb_width = 3
symb_height = 8

# Now split into two atlas char arrays, one for the characters at 6x8 pixels and one for symbols at 3x8
# Note: the symbol atlas is not used in the current implementation, as we know where all the symbols should be and can fail the parse in the cases when we are wrong.
char_atlas = atlas_image[0][:8, :]

chars = char_atlas.shape[1] // char_width

# Reshape into (8, chars, char_width)
char_atlas_chars = char_atlas.reshape(8, chars, char_width).transpose(1, 0, 2).astype(np.float32).compute()
char_glyphs = dict(zip(atlas_chars, char_atlas_chars))

templates = np.stack([char_glyphs[ch] for ch in atlas_chars]).astype(np.float32)
templates = templates - templates.mean(axis=(1, 2), keepdims=True)
template_norms = np.sqrt(np.sum(templates * templates, axis=(1, 2))) + 1e-6

In [4]:
def score_patch_against_template(patch, template, eps=1e-6):
    """
    Returns a normalized correlation score between a patch and a glyph template.
    Higher is better.
    """
    patch = np.asarray(patch, dtype=np.float32)
    template = np.asarray(template, dtype=np.float32)

    patch = patch - patch.mean()
    template = template - template.mean()

    denom = np.sqrt(np.sum(patch ** 2) * np.sum(template ** 2)) + eps
    return float(np.sum(patch * template) / denom)


def classify_at_cursor(image, x, y, atlas, atlas_chars, width, height, eps=1e-6):
    """
    Classify the glyph centered/anchored at a known cursor position.

    Parameters
    ----------
    image : ndarray
        2D grayscale image.
    x, y : int
        Top-left position of the glyph box in the image.
    atlas : dict[str, ndarray]
        Mapping from character label to glyph bitmap.
    atlas_chars : iterable[str]
        Characters to test.
    width, height : int
        Glyph size.
    eps : float
        Small constant for numerical stability.

    Returns
    -------
    best_char : str
    best_score : float
    scores : dict[str, float]
    """
    patch = image[y:y + height, x:x + width]
    if patch.shape != (height, width):
        raise ValueError(f"Patch shape {patch.shape} does not match {(height, width)}")

    scores = {}
    for ch in atlas_chars:
        scores[ch] = score_patch_against_template(patch, atlas[ch], eps=eps)

    best_char = max(scores, key=scores.get)
    best_score = scores[best_char]
    return best_char, best_score, scores



Now that we have all the glyphs set up, we can classify the characters in the aliased-text images

In [5]:
best_char, best_score, scores = classify_at_cursor(
    image=red.astype(np.float32),
    x=5+char_width*1,
    y=6,
    atlas=char_glyphs,
    atlas_chars=char_glyphs.keys(),
    width=6,
    height=8,
)
best_char, best_score, scores

('0',
 0.9998062252998352,
 {'0': 0.9998062252998352,
  '1': -0.08059922605752945,
  '2': 0.40603744983673096,
  '3': 0.681036114692688,
  '4': -0.09153787791728973,
  '5': 0.5023139119148254,
  '6': 0.7777053713798523,
  '7': -0.05396760627627373,
  '8': 0.7773357033729553,
  '9': 0.780293345451355,
  's': 0.20348623394966125})

Below we can see that this causes issues with the antialiased images.

In [6]:
images_2019 = imread(r'/home/rainybyte/AllSkyImages/2019-09/*.JPG')
red_2019 = images_2019.transpose(0, 3, 1, 2)[0][0]

best_char, best_score, scores = classify_at_cursor(
    image=red_2019.astype(np.float32),
    x=5+(char_width)*1,
    y=6,
    atlas=char_glyphs,
    atlas_chars=char_glyphs.keys(),
    width=6,
    height=8,
)
best_char, best_score, scores

('5',
 0.36639389395713806,
 {'0': 0.33899861574172974,
  '1': -0.06114104762673378,
  '2': 0.2295859158039093,
  '3': 0.23315724730491638,
  '4': -0.165544331073761,
  '5': 0.36639389395713806,
  '6': 0.29352709650993347,
  '7': 0.020206689834594727,
  '8': 0.2833721339702606,
  '9': 0.25677579641342163,
  's': 0.02235008217394352})

In [7]:
def classify_patch_fast(patch, templates, template_norms, chars):
    patch = patch.astype(np.float32, copy=False)
    patch = patch - patch.mean()
    patch_norm = np.sqrt(np.sum(patch * patch)) + 1e-6

    # flatten once
    p = patch.ravel()
    t = templates.reshape(templates.shape[0], -1)

    scores = (t @ p) / (template_norms * patch_norm)
    idx = np.argmax(scores)
    return chars[idx], float(scores[idx]), scores

def classify_date_string_fast(
        image,
        tolerance=0.85,
):
    image = image.reshape((480,640))
    DATE_POS = [
        (5 + 0 * char_width, 6),
        (5 + 1 * char_width, 6),
        (5 + 2 * char_width, 6),
        (5 + 3 * char_width, 6),
        (5 + 4 * char_width + symb_width, 6),
        (5 + 5 * char_width + symb_width, 6),
        (5 + 6 * char_width + 2 * symb_width, 6),
        (5 + 7 * char_width + 2 * symb_width, 6),
    ]
    matched_chars = []
    for (x, y) in DATE_POS:
        patch = image[y:y + char_height, x:x + char_width]
        best_char, best_score, scores = classify_patch_fast(patch, templates, template_norms, atlas_chars)
        if best_score < tolerance:
            best_char = '?'
            best_score = None
        matched_chars.append(best_char)
    matched_chars = matched_chars[:4] + ['/'] + matched_chars[4:6] + ['/'] + matched_chars[6:]
    return ''.join(matched_chars)

def classify_date_string_fast_block(block):
    results = []
    for img in block:
        results.append(classify_date_string_fast(img))
    return np.array(results, dtype=str)

In [8]:
classify_date_string_fast(red)

'2010/08/11'

In [9]:
batch = images.transpose(0, 3, 1, 2)[1000:1025]
batch_reds = batch[:,0]

In [10]:
[classify_date_string_fast(image) for image in batch_reds]

['2010/08/15',
 '2010/08/15',
 '2010/08/15',
 '2010/08/15',
 '2010/08/15',
 '2010/08/15',
 '2010/08/15',
 '2010/08/15',
 '2010/08/15',
 '2010/08/15',
 '2010/08/15',
 '2010/08/15',
 '2010/08/15',
 '2010/08/15',
 '2010/08/15',
 '2010/08/15',
 '2010/08/15',
 '2010/08/15',
 '2010/08/15',
 '2010/08/15',
 '2010/08/15',
 '2010/08/15',
 '2010/08/15',
 '2010/08/15',
 '2010/08/15']

In [11]:
batch_reds.map_blocks(classify_date_string_fast_block, dtype=str, drop_axis=(1,2)).compute()

array(['2010/08/15', '2010/08/15', '2010/08/15', '2010/08/15',
       '2010/08/15', '2010/08/15', '2010/08/15', '2010/08/15',
       '2010/08/15', '2010/08/15', '2010/08/15', '2010/08/15',
       '2010/08/15', '2010/08/15', '2010/08/15', '2010/08/15',
       '2010/08/15', '2010/08/15', '2010/08/15', '2010/08/15',
       '2010/08/15', '2010/08/15', '2010/08/15', '2010/08/15',
       '2010/08/15'], dtype='<U10')

In [15]:
all_batch = images.transpose(0, 3, 1, 2)
all_batch_reds = all_batch[:,0]

In [16]:
all_parsed: np.ndarray = all_batch_reds.map_blocks(classify_date_string_fast_block, dtype=object, drop_axis=(1,2), chunks=(32,)).compute().astype(str)
all_parsed

array(['2010/08/11', '2010/08/11', '2010/08/11', ..., '2010/09/01',
       '2010/09/01', '2010/09/01'], shape=(8595,), dtype='<U10')

In [17]:
mask = np.char.find(all_parsed, '?') != -1
pd.DataFrame(all_parsed[mask])

,0
0,2010/0?/13
1,2010/08/1?
2,???0/08/?5
3,2010/0?/16
4,2010/08/?6
5,2010/08/?6
6,2010/08/?6
7,2010/08/1?
8,?010/0?/1?
9,?010/0?/1?
